# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!


  1. Use RAGAS to Generate Synthetic Data
  2. Load them into a LangSmith Dataset
  3. Evaluate our RAG chain against the synthetic test data
  4. Make changes to our pipeline
  5. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gabri\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gabri\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"PSI - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "bills/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying SummaryExtractor:   0%|          | 0/16 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node 372dc168-dd71-4970-affc-6d33027d8e19 does not have a summary. Skipping filtering.
Node 51cbec73-5ca2-4967-9b08-f97c9ffe66f8 does not have a summary. Skipping filtering.
Node 63b46daa-d4de-456d-a321-3051155d58ce does not have a summary. Skipping filtering.
Node 4b32d0f1-94a9-41bf-8945-8b784518726f does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/56 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 20, relationships: 137)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("bills/ai_law.json")
bills_data_kg = KnowledgeGraph.load("bills/ai_law.json")
bills_data_kg

KnowledgeGraph(nodes: 20, relationships: 137)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=bills_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

</div>


<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer :

* `SingleHopSpecificQuerySynthesizer` generates specific, single-hop questions that can be answered directly from a single document or source. `MultiHopAbstractQuerySynthesizer` and `MultiHopSpecificQuerySynthesizer` both generate multi-hop questions that require multi-step reasoning across multiple documents or sources but the former requires broader and more interpretive responses while the latter is more factual, direct, and clear. 
</span>


Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the role of the REPUBLIC OF THE PHILIP...,[TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE P...,The context discusses a legislative act introd...,single_hop_specifc_query_synthesizer
1,Philippines AI good or bad?,[AI presents enormous opportunities for the Ph...,AI presents enormous opportunities for the Phi...,single_hop_specifc_query_synthesizer
2,What was the TWENTIETH CONGRESS?,[TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE P...,The TWENTIETH CONGRESS OF THE \nREPUBLIC OF TH...,single_hop_specifc_query_synthesizer
3,What is the role of Artificiai Inteiiigence in...,"[1 \na) Promote innovation, technological adva...",Artificiai Inteiiigence (AI) refers to systems...,single_hop_specifc_query_synthesizer
4,What AGI means in AI policy?,[1\n2\n3\n4\n5\n6\n7\n8 \n9\n10\n11\n12\n13\n1...,Artificial General Intelligence (AGI) refers t...,single_hop_specifc_query_synthesizer
5,How do rules and regulations for enforcement e...,"[<1-hop>\n\n1 \nendorsements, voice recordings...",The context specifies that within ninety days ...,multi_hop_abstract_query_synthesizer
6,How do the audit mechanisms and sanctions for ...,[<1-hop>\n\n1 \nSec. 15. AI Ethics Review Boar...,Sec. 15 of the AI Ethics Review Board establis...,multi_hop_abstract_query_synthesizer
7,How does the Philippine AI regulation framewor...,[<1-hop>\n\n1 \nk) Issue advisory opinions on ...,"The Philippine AI regulation framework, as out...",multi_hop_abstract_query_synthesizer
8,"How does the AI Policy Act, which includes the...",[<1-hop>\n\n1\n2\n3\n4\n5\n6\n7\n8 \n9\n10\n11...,"The AI Policy Act, encompassing the Artificial...",multi_hop_specific_query_synthesizer
9,How DOST involved in AI and DOST do they do st...,[<1-hop>\n\n1 \niii) \nProof of employer engag...,The context shows that DOST is involved in est...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying SummaryExtractor:   0%|          | 0/16 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node 021d6f49-84ed-4a73-b09e-06d1fd0ed022 does not have a summary. Skipping filtering.
Node 9d32451a-cdf4-4d0f-8993-23a395b6e219 does not have a summary. Skipping filtering.
Node 7684475d-ac4f-4aea-b011-3cfa48166ad0 does not have a summary. Skipping filtering.
Node f309a075-dfa9-44a1-85a1-56f0adddbb7e does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/56 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Who is PIA S. CAYETANO in the context of AI po...,[TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE P...,PIA S. CAYETANO is a senator who introduced an...,single_hop_specifc_query_synthesizer
1,What dangers of Artificial Superintelligence a...,[AI presents enormous opportunities for the Ph...,The context mentions that the rise of Artifici...,single_hop_specifc_query_synthesizer
2,What does Section 10 of the Artificial Intelli...,[TWENTIETH CONGRESS OF THE \nREPUBLIC OF THE P...,The provided context does not include specific...,single_hop_specifc_query_synthesizer
3,What AI do policy do?,"[1 \na) Promote innovation, technological adva...",AI refers to systems that allow machines to th...,single_hop_specifc_query_synthesizer
4,How does responsible ethical AI use relate to ...,"[<1-hop>\n\n1 \na) Promote innovation, technol...","The context emphasizes promoting responsible, ...",multi_hop_abstract_query_synthesizer
5,How do capacity-blding efforts for inclusive g...,[<1-hop>\n\n1 \niii) \nProof of employer engag...,The context highlights that capacity-building ...,multi_hop_abstract_query_synthesizer
6,How does the NAIC's creation of an AI Ethics R...,[<1-hop>\n\n1 \nSec. 8. NAICSecretariat. - The...,The NAIC's creation of an AI Ethics Review Boa...,multi_hop_abstract_query_synthesizer
7,How do penalties for non-disclosure and mislab...,[<1-hop>\n\n1 \niii) Mandatory compliance trai...,The penalties for failure to disclose AI-gener...,multi_hop_abstract_query_synthesizer
8,How do policies addressing AI safety and ethic...,[<1-hop>\n\n1\n2\n3\n4\n5\n6\n7\n8 \n9\n10\n11...,The policies on AI safety and ethics recognize...,multi_hop_specific_query_synthesizer
9,How do the DICT and the NAIC collaborate to en...,"[<1-hop>\n\n1 \nendorsements, voice recordings...",The DICT is responsible for promulgating imple...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Philippines AI Bills"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Philippines AI Bills"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Bills RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "How much is the penalty for spreading disinformation?"})

'The penalty for spreading disinformation using AI is a fine of One Million Pesos (Php 1,000,000) to Five Million Pesos (Php 5,000,000), or imprisonment of three (3) years to ten (10) years, or both, at the discretion of the court.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### 🏗️ Question #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer:

* `qa_evaluator` measures the correctness or accuracy of the model’s predicted answer by comparing it to the correct reference.
`labeled_helpfulness_evaluator` evaluates whether the response is useful or insightful for the user in context of the correct answer.

</div>

## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'bold-scene-19' at:
https://smith.langchain.com/o/1a05e942-5d81-4e93-961a-53ef9c8c2468/datasets/9d749867-b3e9-4b1d-8388-ed1cf8aed364/compare?selectedSessions=f8a45e15-45a4-48c6-a266-ea1dbae57fc2




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How does TESDA contribute to AI regulation and...,I don't know.,None,"TESDA is part of the NAIC's composition, as ou...",0,0,0,4.143958,03b07308-718c-49fc-b36c-fac13441f14b,09126def-f2b5-40a7-a959-d92d936b06cf
1,Php 2000000 or Php 500000 use for AI law or so...,"Based on the provided context, the amounts of ...",None,The context mentions penalties related to AI a...,1,1,0,8.035161,b6994340-ea4d-437d-9ef6-93f3285f4018,363ecb78-1c77-4e60-a252-ceae7bb3078e
2,How do the DICT and the NAIC collaborate to en...,I don't know.,None,The DICT is responsible for promulgating imple...,0,0,0,2.927196,051cd401-105e-4341-9551-9fced54876ee,fb0efc68-8329-40c8-aa84-b5a3a78eadcc
3,How do policies addressing AI safety and ethic...,Policies addressing AI safety and ethics incor...,None,The policies on AI safety and ethics recognize...,1,1,0,5.410006,6fe1f9fb-a3e0-4b7c-9746-079c246ed0a5,312d30ab-f866-4619-8c86-c2da287d4ed8
4,How do penalties for non-disclosure and mislab...,"Based on the provided context, penalties for n...",None,The penalties for failure to disclose AI-gener...,1,1,0,7.774252,27891a8d-0e23-4510-8b73-8982767a1d6d,109dfcee-6283-409f-9f92-4276807ed097
5,How does the NAIC's creation of an AI Ethics R...,The NAIC's creation of an AI Ethics Review Boa...,None,The NAIC's creation of an AI Ethics Review Boa...,1,1,0,4.593466,72deeecb-ec84-4771-bb54-1178487803ec,cae5aa68-7bde-4f1c-a775-c06729d9cf4c
6,How do capacity-blding efforts for inclusive g...,Capacity-building efforts for inclusive growth...,None,The context highlights that capacity-building ...,1,0,0,4.104522,de4987d2-cdfd-44d7-b246-b4a146c6bedb,6fcde36e-6e79-4a06-ba04-73b5396e543e
7,How does responsible ethical AI use relate to ...,"Based on the provided context, responsible and...",None,"The context emphasizes promoting responsible, ...",1,1,0,6.542787,e4b50016-be26-478b-bbea-b324d33d019a,abff42fb-7abc-489e-903e-a2bd841bfa66
8,What AI do policy do?,"Based on the provided context, the AI policy a...",None,AI refers to systems that allow machines to th...,0,1,0,8.289417,5874d0fd-1f8d-4cfa-9c97-07d2d5ac45ec,e92989cf-2978-427a-a68d-28c902e3372d
9,What does Section 10 of the Artificial Intelli...,I don't know.,None,The provided context does not include specific...,1,0,0,7.365325,84fdc45c-56b1-49b1-bd38-5bdf1f24da4f,af7a4118-6cea-4038-9adb-71224d5f5cda


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer: 

Chunk size can affect the retrieval accuracy, generation quality, embedding quality, cost, and latency of the application. A chunk size too small may potentially lack context and information, require more embedding computation time and storage, and have slower retrieval. And while a larger chunk size can have faster retrieval, a chunk size too large may contain too much noise, and may require more tokens thus increasing cost. 

</div>

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Answer :

The quality of the embedding model directly impacts the quality of the retrieval system. If the model captures meaning well, it will retrieve highly relevant documents which will provide a better context for the LLM's response, resulting into better LLM outputs. If the application is domain/task-specific, choosing a corresponding domain/task-tuned embedding model will also improve performance. Embedding models also differ in terms of size, latency. These factors may make or break the application. 

</div>

In [34]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Bills RAG 2"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
empathy_rag_chain.invoke({"question" : "Why is the Philippines AI Bill important?"})

'Thank you for your thoughtful question. Based on the context provided, the Philippines AI Bill is important because it aims to carefully balance encouraging technological innovation with ensuring that AI systems remain safe, ethical, transparent, and under meaningful human oversight. It recognizes the transformative potential of AI while addressing the need to protect the rights and welfare of every citizen. \n\nThe bill is designed to provide a responsible and lawful framework to guide AI development during its early stages, supporting Filipino ingenuity and national development. It also underscores the State’s responsibility to prevent AI from being used to commit crimes, violate rights, or cause harm—whether intentionally or accidentally. Ultimately, this bill envisions a future where AI contributes positively to society while safeguarding the Filipino people, which makes it truly vital.\n\nI hope this helps you understand the significance of the bill. If you have more questions or

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'helpful-popcorn-83' at:
https://smith.langchain.com/o/1a05e942-5d81-4e93-961a-53ef9c8c2468/datasets/9d749867-b3e9-4b1d-8388-ed1cf8aed364/compare?selectedSessions=4e163cb6-fb9d-4893-b4a9-3e61983c26d6




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How does TESDA contribute to AI regulation and...,Thank you for your thoughtful question. Based ...,None,"TESDA is part of the NAIC's composition, as ou...",1,1,1,14.110575,03b07308-718c-49fc-b36c-fac13441f14b,dc1d93cc-856c-4c1d-bb16-ddd7ecda419f
1,Php 2000000 or Php 500000 use for AI law or so...,Thank you for your question. Based on the cont...,None,The context mentions penalties related to AI a...,1,1,1,9.064656,b6994340-ea4d-437d-9ef6-93f3285f4018,c8b023d6-ca8f-4a04-a1ed-0c510542f40b
2,How do the DICT and the NAIC collaborate to en...,Thank you for your thoughtful question. Based ...,None,The DICT is responsible for promulgating imple...,1,1,1,10.260118,051cd401-105e-4341-9551-9fced54876ee,4c60916e-bab9-44df-86f8-a513f93e745a
3,How do policies addressing AI safety and ethic...,Thank you for your thoughtful question about h...,None,The policies on AI safety and ethics recognize...,1,1,1,11.965109,6fe1f9fb-a3e0-4b7c-9746-079c246ed0a5,06c821a9-1c4e-4fe1-97b5-a5998db24603
4,How do penalties for non-disclosure and mislab...,Thank you for your thoughtful question. Based ...,None,The penalties for failure to disclose AI-gener...,1,1,1,11.638028,27891a8d-0e23-4510-8b73-8982767a1d6d,fcfd0523-36a0-4db1-b065-ba9bf6c467f8
5,How does the NAIC's creation of an AI Ethics R...,Thank you for your thoughtful question. Based ...,None,The NAIC's creation of an AI Ethics Review Boa...,1,1,1,5.743420,72deeecb-ec84-4771-bb54-1178487803ec,1f9570b5-7e72-4902-bf77-0764e8c1c608
6,How do capacity-blding efforts for inclusive g...,Thank you for your thoughtful question about h...,None,The context highlights that capacity-building ...,1,1,1,6.955523,de4987d2-cdfd-44d7-b246-b4a146c6bedb,f9c0bb9c-adec-4534-8d28-419dffa84ef2
7,How does responsible ethical AI use relate to ...,Thank you for your thoughtful question. Based ...,None,"The context emphasizes promoting responsible, ...",0,0,1,6.522272,e4b50016-be26-478b-bbea-b324d33d019a,e4ab4487-3c1c-4642-bf37-c1b25285b203
8,What AI do policy do?,Thank you for your question. From the context ...,None,AI refers to systems that allow machines to th...,0,1,1,8.020949,5874d0fd-1f8d-4cfa-9c97-07d2d5ac45ec,6509d10d-529e-46c8-a5bd-f20729cfe338
9,What does Section 10 of the Artificial Intelli...,Thank you for your thoughtful question about S...,None,The provided context does not include specific...,1,1,1,6.501649,84fdc45c-56b1-49b1-bd38-5bdf1f24da4f,845eba81-09e6-4d90-bd51-29ac6e01b0af


<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

#### 🏗️ Question #4:

Explain why you believe certain metrics changed in certain ways, and provide a screenshot of the difference between the two chains.

</div>

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px; ">

### Answer:

Using a larger chunk size allowed more context in each chunk, allowing the model to retrieve more complete information, improving correctness and helpfulness. These metrics were further improved by leveraging a better embedding model that better captured semantic meaning thus allowed higher relevance of retrieved documents. Lastly, adding the empathy-focused prompt guided the model to respond with kindness and emotional understanding, greatly increasing the empathy score. 



<img src="langsmith-chart.png" width="500"/>

<img src="langsmith-sdg1.png" width="800"/>

<img src="langsmith-sdg2.png" width="800"/>

</div>